<a href="https://colab.research.google.com/github/pkuzzw/2019/blob/master/MOE_Version1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 在第一个cell中执行 - 安装依赖
!pip install transformers torch accelerate lm_eval
!pip install datasets peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 7.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0b22309728b752cb72e2f00ef823a53392be2797a15644edad5e8c9344bd0dcb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c6

In [ ]:
!pip install GPUtil

  Preparing metadata (setup.py) ... done
  Created wheel for GPUtil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=5b35d6162ec0bcff049f289896189dce7e61c9fe849f68390dc8f6df8abac6e6
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built GPUtil


In [ ]:
# 第二个cell - 导入必要的库
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import psutil
import GPUtil
from datetime import datetime
import pandas as pd
import numpy as np

# 检查GPU可用性
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch版本: 2.8.0+cu126
CUDA可用: True
GPU设备: NVIDIA A100-SXM4-80GB
GPU内存: 79.3 GB


In [ ]:
# 第三个cell - 加载模型和分词器
model_name = "Qwen/Qwen1.5-MoE-A2.7B"
print(f"开始加载模型: {model_name}")
print(f"开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 记录开始时的GPU内存
if torch.cuda.is_available():
    initial_memory = torch.cuda.memory_allocated() / 1024**3
    print(f"加载前GPU内存使用: {initial_memory:.2f} GB")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 加载模型 - 使用bfloat16以节省显存
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("模型加载完成!")
print(f"完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 检查加载后的内存使用
if torch.cuda.is_available():
    final_memory = torch.cuda.memory_allocated() / 1024**3
    print(f"加载后GPU内存使用: {final_memory:.2f} GB")
    print(f"模型占用显存: {final_memory - initial_memory:.2f} GB")

开始加载模型: Qwen/Qwen1.5-MoE-A2.7B
开始时间: 2025-10-12 08:44:17
加载前GPU内存使用: 0.00 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

... (10 outputs left out)

model-00002-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

模型加载完成!
完成时间: 2025-10-12 08:47:15
加载后GPU内存使用: 26.67 GB
模型占用显存: 26.67 GB


In [ ]:
# 第四个cell - 基础功能测试
def test_basic_functionality():
    test_prompts = [
        "中国的首都是",
        "请用Python写一个Hello World程序:",
        "机器学习是",
        "The capital of France is"
    ]

    results = []

    for prompt in test_prompts:
        print(f"\n{'='*50}")
        print(f"测试提示: {prompt}")
        print(f"{'='*50}")

        # 编码输入
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # 生成回复
        with torch.no_grad():
            start_time = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                temperature=0.1
            )
            end_time = time.time()

        # 解码输出
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = generated_text[len(prompt):]

        generation_time = end_time - start_time
        num_tokens = len(outputs[0]) - len(inputs.input_ids[0])

        print(f"生成内容: {response}")
        print(f"生成时间: {generation_time:.2f}s")
        print(f"生成token数: {num_tokens}")
        print(f"速度: {num_tokens/generation_time:.2f} tokens/s")

        results.append({
            'prompt': prompt,
            'response': response,
            'time': generation_time,
            'tokens': num_tokens,
            'speed': num_tokens/generation_time
        })

    return results

# 执行基础测试
basic_results = test_basic_functionality()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



测试提示: 中国的首都是
生成内容: ____
A. 北京
B. 上海
C. 天津
D. 重庆
答案:
A
生成时间: 3.63s
生成token数: 28
速度: 7.71 tokens/s

测试提示: 请用Python写一个Hello World程序:
生成内容:  print("Hello World")
生成时间: 0.63s
生成token数: 6
速度: 9.47 tokens/s

测试提示: 机器学习是
生成内容: 人工智能的一个分支，它使计算机能够从数据中学习，而无需明确编程。机器学习算法可以自动识别模式并进行预测，从而帮助人们做出更好的决策。机器学习在许多领域都有应用，包括自然语言处理、计算机
生成时间: 3.64s
生成token数: 50
速度: 13.73 tokens/s

测试提示: The capital of France is
生成内容:  ______.
Paris
London
Berlin
Rome
答案:
A
生成时间: 1.19s
生成token数: 15
速度: 12.64 tokens/s


In [ ]:
# 第五个cell - 准备测试数据集
test_datasets = {
    "知识问答": [
        "中国的首都是哪个城市？",
        "太阳系中最大的行星是什么？",
        "谁写了《哈姆雷特》？",
        "水的化学式是什么？",
        "爱因斯坦最著名的公式是什么？"
    ],
    "数学推理": [
        "如果小明有5个苹果，小红给了他3个苹果，然后他吃了2个，现在小明有几个苹果？",
        "计算15乘以24等于多少？",
        "一个正方形的边长是8厘米，它的面积是多少？",
        "30的百分之20是多少？"
    ],
    "代码生成": [
        "用Python写一个函数计算斐波那契数列",
        "写一个JavaScript函数来反转字符串",
        "用Python实现快速排序算法",
        "写一个函数判断一个数是否为质数"
    ],
    "逻辑推理": [
        "如果所有猫都是动物，而且有些动物是宠物，那么有些猫是宠物吗？为什么？",
        "如果A大于B，B大于C，那么A和C的关系是什么？"
    ]
}

In [ ]:
# 第六个cell - 性能测试函数
def comprehensive_benchmark(datasets, num_runs=3):
    benchmark_results = {}

    for category, prompts in datasets.items():
        print(f"\n{'='*60}")
        print(f"测试类别: {category}")
        print(f"{'='*60}")

        category_results = []

        for prompt in prompts:
            print(f"\n测试提示: {prompt}")

            run_times = []
            token_speeds = []
            memory_usages = []

            for run in range(num_runs):
                # 清理GPU缓存
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats()

                # 编码输入
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

                # 生成回复
                with torch.no_grad():
                    start_time = time.time()
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=100,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        temperature=0.1
                    )
                    end_time = time.time()

                # 计算指标
                generation_time = end_time - start_time
                num_input_tokens = len(inputs.input_ids[0])
                num_output_tokens = len(outputs[0])
                num_new_tokens = num_output_tokens - num_input_tokens

                tokens_per_second = num_new_tokens / generation_time if generation_time > 0 else 0

                # 获取内存使用
                if torch.cuda.is_available():
                    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
                else:
                    peak_memory = 0

                # 解码输出
                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                response = generated_text[len(prompt):].strip()

                run_times.append(generation_time)
                token_speeds.append(tokens_per_second)
                memory_usages.append(peak_memory)

                if run == 0:  # 只在第一次运行时显示完整回复
                    print(f"模型回复: {response}")

            # 计算统计信息
            avg_time = np.mean(run_times)
            avg_speed = np.mean(token_speeds)
            avg_memory = np.mean(memory_usages)

            print(f"平均生成时间: {avg_time:.2f}s")
            print(f"平均生成速度: {avg_speed:.2f} tokens/s")
            print(f"平均峰值显存: {avg_memory:.2f} GB")

            category_results.append({
                'prompt': prompt,
                'avg_time': avg_time,
                'avg_speed': avg_speed,
                'avg_memory': avg_memory,
                'response': response
            })

        benchmark_results[category] = category_results

    return benchmark_results

# 执行综合基准测试
print("开始综合基准测试...")
benchmark_results = comprehensive_benchmark(test_datasets)

开始综合基准测试...

测试类别: 知识问答

测试提示: 中国的首都是哪个城市？
模型回复: ____
A. 北京
B. 上海
C. 广州
D. 深圳
答案:
A
平均生成时间: 2.26s
平均生成速度: 13.28 tokens/s
平均峰值显存: 26.68 GB

测试提示: 太阳系中最大的行星是什么？
模型回复: 太阳系中最大的行星是木星。
平均生成时间: 1.00s
平均生成速度: 12.03 tokens/s
平均峰值显存: 26.68 GB

测试提示: 谁写了《哈姆雷特》？
模型回复: ____
A. 莎士比亚
B. 但丁
C. 莫里哀
D. 塞万提斯
答案:
A
平均生成时间: 2.80s
平均生成速度: 13.21 tokens/s
平均峰值显存: 26.68 GB

测试提示: 水的化学式是什么？
模型回复: 水的化学式是什么？ 水的化学式是H2O。
平均生成时间: 1.42s
平均生成速度: 12.71 tokens/s
平均峰值显存: 26.68 GB

测试提示: 爱因斯坦最著名的公式是什么？
模型回复: ____
A. E=mc2
B. E=mc
C. E=mc3
D. E=mc4
答案:
A
平均生成时间: 2.52s
平均生成速度: 13.11 tokens/s
平均峰值显存: 26.68 GB

测试类别: 数学推理

测试提示: 如果小明有5个苹果，小红给了他3个苹果，然后他吃了2个，现在小明有几个苹果？
模型回复: 小明现在有2个苹果。

好的，那请你告诉我，如果小明再吃掉1个苹果，他现在有几个苹果？ 如果小明再吃掉1个苹果，他现在就只有1个苹果了。
平均生成时间: 3.90s
平均生成速度: 12.82 tokens/s
平均峰值显存: 26.69 GB

测试提示: 计算15乘以24等于多少？
模型回复: 15乘以24等于360。
平均生成时间: 1.11s
平均生成速度: 11.71 tokens/s
平均峰值显存: 26.68 GB

测试提示: 一个正方形的边长是8厘米，它的面积是多少？
模型回复: 64平方厘米。
平均生成时间: 0.76s
平均生成速度: 9.21 tokens/s
平均峰值显存: 26.68 GB

测试提示: 30的百分之20是多少？
模型回复: 6
平均生成时间: 0.35s
平均生成

In [ ]:
# 第七个cell - 效率基准测试
def efficiency_benchmark():
    print("开始效率基准测试...")

    # 测试不同的输入输出长度
    test_cases = [
        {"input_len": 50, "output_len": 50, "desc": "短文本生成"},
        {"input_len": 200, "output_len": 100, "desc": "中长文本生成"},
        {"input_len": 50, "output_len": 200, "desc": "长输出生成"}
    ]

    efficiency_results = []

    for test_case in test_cases:
        print(f"\n测试场景: {test_case['desc']}")

        # 生成测试文本
        test_text = "这是一段测试文本。" * (test_case['input_len'] // 10)
        inputs = tokenizer(test_text, return_tensors="pt").to(model.device)

        # 预热
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=10)

        # 正式测试
        start_time = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=test_case['output_len'],
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        end_time = time.time()

        # 计算指标
        total_time = end_time - start_time
        input_tokens = len(inputs.input_ids[0])
        output_tokens = len(outputs[0])
        new_tokens = output_tokens - input_tokens

        tokens_per_second = new_tokens / total_time

        # 内存使用
        if torch.cuda.is_available():
            peak_memory = torch.cuda.max_memory_allocated() / 1024**3
        else:
            peak_memory = 0

        result = {
            'scenario': test_case['desc'],
            'input_tokens': input_tokens,
            'output_tokens': new_tokens,
            'total_time': total_time,
            'tokens_per_second': tokens_per_second,
            'peak_memory_gb': peak_memory
        }

        efficiency_results.append(result)

        print(f"输入token数: {input_tokens}")
        print(f"输出token数: {new_tokens}")
        print(f"总时间: {total_time:.2f}s")
        print(f"生成速度: {tokens_per_second:.2f} tokens/s")
        print(f"峰值显存: {peak_memory:.2f} GB")

    return efficiency_results

# 执行效率测试
efficiency_results = efficiency_benchmark()

开始效率基准测试...

测试场景: 短文本生成
输入token数: 30
输出token数: 50
总时间: 3.76s
生成速度: 13.30 tokens/s
峰值显存: 26.69 GB

测试场景: 中长文本生成
输入token数: 120
输出token数: 100
总时间: 7.48s
生成速度: 13.37 tokens/s
峰值显存: 26.72 GB

测试场景: 长输出生成
输入token数: 30
输出token数: 200
总时间: 14.60s
生成速度: 13.70 tokens/s
峰值显存: 26.72 GB


In [ ]:
# 第八个cell - 结果汇总
def summarize_results(benchmark_results, efficiency_results):
    print("\n" + "="*80)
    print("基准测试结果汇总")
    print("="*80)

    # 性能结果汇总
    print("\n📊 性能测试结果:")
    print("-" * 50)

    all_speeds = []
    all_memories = []

    for category, results in benchmark_results.items():
        category_speeds = [r['avg_speed'] for r in results]
        category_memories = [r['avg_memory'] for r in results]

        avg_speed = np.mean(category_speeds)
        avg_memory = np.mean(category_memories)

        all_speeds.extend(category_speeds)
        all_memories.extend(category_memories)

        print(f"{category:12} | 平均速度: {avg_speed:6.2f} tokens/s | 平均显存: {avg_memory:5.2f} GB")

    # 效率结果汇总
    print("\n⚡ 效率测试结果:")
    print("-" * 50)
    for result in efficiency_results:
        print(f"{result['scenario']:15} | 速度: {result['tokens_per_second']:6.2f} tokens/s | 显存: {result['peak_memory_gb']:5.2f} GB")

    # 总体统计
    print("\n📈 总体统计:")
    print("-" * 50)
    print(f"平均生成速度: {np.mean(all_speeds):.2f} tokens/s")
    print(f"平均峰值显存: {np.mean(all_memories):.2f} GB")
    print(f"最大峰值显存: {np.max(all_memories):.2f} GB")

    # 硬件信息
    print("\n🖥️  硬件信息:")
    print("-" * 50)
    if torch.cuda.is_available():
        gpu_props = torch.cuda.get_device_properties(0)
        print(f"GPU: {gpu_props.name}")
        print(f"GPU内存: {gpu_props.total_memory / 1024**3:.1f} GB")
        print(f"已用GPU内存: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"剩余GPU内存: {(gpu_props.total_memory - torch.cuda.memory_allocated()) / 1024**3:.2f} GB")

# 汇总结果
summarize_results(benchmark_results, efficiency_results)


基准测试结果汇总

📊 性能测试结果:
--------------------------------------------------
知识问答         | 平均速度:  12.87 tokens/s | 平均显存: 26.68 GB
数学推理         | 平均速度:  10.55 tokens/s | 平均显存: 26.68 GB
代码生成         | 平均速度:  13.70 tokens/s | 平均显存: 26.69 GB
逻辑推理         | 平均速度:  12.10 tokens/s | 平均显存: 26.68 GB

⚡ 效率测试结果:
--------------------------------------------------
短文本生成           | 速度:  13.30 tokens/s | 显存: 26.69 GB
中长文本生成          | 速度:  13.37 tokens/s | 显存: 26.72 GB
长输出生成           | 速度:  13.70 tokens/s | 显存: 26.72 GB

📈 总体统计:
--------------------------------------------------
平均生成速度: 12.37 tokens/s
平均峰值显存: 26.69 GB
最大峰值显存: 26.70 GB

🖥️  硬件信息:
--------------------------------------------------
GPU: NVIDIA A100-SXM4-80GB
GPU内存: 79.3 GB
已用GPU内存: 26.67 GB
剩余GPU内存: 52.64 GB


In [ ]:
# 第九个cell - 保存结果
def save_results(benchmark_results, efficiency_results, filename="baseline_results.json"):
    import json

    # 准备保存的数据
    save_data = {
        "timestamp": datetime.now().isoformat(),
        "model": "Qwen1.5-MoE-A2.7B",
        "hardware": {
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "gpu_memory_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
        },
        "benchmark_results": benchmark_results,
        "efficiency_results": efficiency_results
    }

    # 保存到文件
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)

    print(f"结果已保存到: {filename}")

    # 同时保存为CSV用于分析
    df_data = []
    for category, results in benchmark_results.items():
        for result in results:
            df_data.append({
                'category': category,
                'prompt': result['prompt'],
                'avg_time': result['avg_time'],
                'avg_speed': result['avg_speed'],
                'avg_memory': result['avg_memory']
            })

    df = pd.DataFrame(df_data)
    df.to_csv('baseline_results.csv', index=False, encoding='utf-8-sig')
    print("CSV结果已保存到: baseline_results.csv")

    return save_data

# 保存结果
saved_results = save_results(benchmark_results, efficiency_results)

结果已保存到: baseline_results.json
CSV结果已保存到: baseline_results.csv


In [ ]:
!nvidia-smi

Sun Oct 12 08:50:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             63W /  400W |   27901MiB /  81920MiB |     26%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----